# Midterm project: predicting online-shopper purchase intent

**Machine Learning Zoomcamp 2026 · DataTalks.Club**

## Problem
An online shop sees thousands of browsing sessions per day, but only a small fraction ends in a purchase.
If the shop can estimate *during or right after a session* how likely the visitor is to buy, it can act on it:
show a discount to a hesitant high-intent visitor, skip the pop-up for someone who is about to buy anyway,
or retarget the people who left without buying.

**Task:** binary classification. Given the behaviour recorded in one session (pages viewed, time spent, bounce/exit rates,
page value, month, traffic source, visitor type, ...), predict whether the session ends with a purchase (`Revenue = 1`).

**Data:** [UCI Online Shoppers Purchasing Intention Dataset](https://archive.ics.uci.edu/dataset/468) (Sakar et al., 2019), 12,330 sessions, 17 features, CC BY 4.0.

**Evaluation:** only a small share of sessions convert, so accuracy is misleading (always predicting "no purchase" is already ~85 % accurate).
I select models on **ROC-AUC** (ranking quality), also report **PR-AUC**, and pick an operating threshold by maximising **F1**.

**Notebook outline**
1. Load and clean the data
2. Exploratory data analysis (distributions, target, conversion rates, correlations, feature importance)
3. Feature engineering
4. Train/test split and baseline models
5. Hyper-parameter tuning
6. Final model, threshold and test-set evaluation
7. Feature importance of the final model
8. Export the tuned parameters for `train.py`

## 1. Setup and data loading
Shared logic (cleaning, feature engineering, pipelines, metrics) lives in `common.py`, so that the notebook, `train.py` and the web service use exactly the same code.

In [ ]:
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from scipy.stats import loguniform, randint, uniform
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    mutual_info_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_val_predict, train_test_split

import common
from download_data import download

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

SEED = common.SEED
TARGET = common.TARGET

In [ ]:
csv_path = download()  # downloads data/online_shoppers_intention.csv (skipped if it already exists)
raw = pd.read_csv(csv_path)
print(raw.shape)
raw.head()

In [ ]:
raw.info()

### Data quality checks
Missing values, duplicates, value ranges and category spelling.

In [ ]:
print("missing values  :", int(raw.isna().sum().sum()))
print("duplicated rows :", int(raw.duplicated().sum()))
print("rates inside [0, 1]            :", bool(raw[common.RATE_COLUMNS].apply(lambda s: s.between(0, 1).all()).all()))
print("counts/durations/values >= 0   :", bool((raw[common.COUNT_COLUMNS + common.DURATION_COLUMNS + ["PageValues"]] >= 0).all().all()))
raw["Month"].value_counts()

**What to notice**
* `Month` uses three-letter abbreviations, except for **June, which is spelled out**, and January and April never occur in the data.
* The file contains **duplicated sessions**. If duplicates end up in both the training and the test split, the test score is inflated, so they are dropped before splitting.
* `Weekend` and `Revenue` are booleans.
* `OperatingSystems`, `Browser`, `Region` and `TrafficType` are integer *codes*. The numbers have no order, so they are treated as categories.

All of these fixes are implemented in `common.clean_dataset()`.

In [ ]:
df = common.clean_dataset(raw)
print(f"{len(raw):,} rows -> {len(df):,} after dropping duplicates")
df["Month"].value_counts().reindex(common.MONTHS).dropna().astype(int)

In [ ]:
raw[common.NUMERIC_RAW].describe().T

## 2. Exploratory data analysis
### 2.1 Target distribution

In [ ]:
target_counts = df[TARGET].value_counts().sort_index()
display(pd.DataFrame({"sessions": target_counts, "share": (target_counts / len(df)).round(3)}))

fig, ax = plt.subplots(figsize=(4.5, 3.5))
ax.bar(["no purchase", "purchase"], target_counts.values, color=["#8da0cb", "#fc8d62"])
ax.set_ylabel("sessions")
ax.set_title("Target distribution")
plt.show()

The classes are imbalanced (roughly 1 purchase in 6 or 7 sessions). That is why I evaluate with ROC-AUC / PR-AUC and tune the decision threshold instead of relying on accuracy.

### 2.2 Numeric features
Page counts, durations and page values are heavily right-skewed and zero-inflated, so histograms are drawn on a `log1p` scale.

In [ ]:
numeric_cols = [c for c in common.NUMERIC_RAW if c != "Weekend"]
labels = df[TARGET].map({0: "no purchase", 1: "purchase"})

fig, axes = plt.subplots(2, 5, figsize=(22, 7))
for ax, col in zip(axes.ravel(), numeric_cols):
    sns.histplot(x=np.log1p(df[col]), hue=labels, bins=40, stat="density", common_norm=False,
                 element="step", ax=ax, legend=(col == numeric_cols[0]))
    ax.set_xlabel(f"log1p({col})")
    ax.set_title(col)
fig.tight_layout()
plt.show()

In [ ]:
zero_share = (df[numeric_cols] == 0).mean().sort_values(ascending=False).round(3)
zero_share.rename("share of zeros").to_frame()

### 2.3 `PageValues`
`PageValues` is the average value of the pages a visitor saw before the transaction (from Google Analytics). Most sessions have a page value of exactly zero, and I expect a strong link with the target. A closer look:

In [ ]:
(df.assign(has_page_value=df["PageValues"] > 0)
   .groupby(TARGET)
   .agg(sessions=("PageValues", "size"),
        mean_page_value=("PageValues", "mean"),
        median_page_value=("PageValues", "median"),
        share_with_page_value=("has_page_value", "mean"))
   .round(3))

In [ ]:
df.groupby(df["PageValues"] > 0)[TARGET].agg(sessions="size", conversion_rate="mean").rename_axis("PageValues > 0").round(3)

### 2.4 Conversion rate by categorical feature
Dashed line = overall conversion rate. Bars far from the line identify categories that carry signal.

In [ ]:
cat_cols = common.CATEGORICAL + ["Weekend"]
overall_rate = df[TARGET].mean()

fig, axes = plt.subplots(2, 4, figsize=(22, 9))
for ax, col in zip(axes.ravel(), cat_cols):
    rates = df.groupby(col)[TARGET].agg(rate="mean", sessions="size")
    if col == "Month":
        rates = rates.reindex(common.MONTHS).dropna()
    ax.bar(rates.index.astype(str), rates["rate"], color="#66c2a5")
    ax.axhline(overall_rate, color="grey", linestyle="--")
    ax.set_title(f"Conversion rate by {col}")
    ax.tick_params(axis="x", rotation=60)
for ax in axes.ravel()[len(cat_cols):]:
    ax.axis("off")
fig.tight_layout()
plt.show()

In [ ]:
display(df.groupby("VisitorType")[TARGET].agg(sessions="size", conversion_rate="mean").round(3))
display(df.groupby("Weekend")[TARGET].agg(sessions="size", conversion_rate="mean").round(3))
df.groupby("Month")[TARGET].agg(sessions="size", conversion_rate="mean").reindex(common.MONTHS).dropna().round(3)

Some categories contain very few sessions (rare traffic types, browsers, the `Other` visitor type), so their conversion rates are noisy. One-hot encoding with `handle_unknown="ignore"` keeps the pipeline robust to such categories.

### 2.5 Correlations

In [ ]:
corr = df[common.NUMERIC_RAW + [TARGET]].corr()
plt.figure(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, linewidths=0.5)
plt.title("Correlation matrix (numeric features and target)")
plt.show()

Pairs such as `ProductRelated` / `ProductRelated_Duration` and `BounceRates` / `ExitRates` are strongly correlated. That is fine for tree-based models and for regularised logistic regression, but it is a reason not to over-interpret individual logistic-regression coefficients.

### 2.6 Univariate feature importance
* **Numeric features:** ROC-AUC of the raw feature used as a score (direction-agnostic: values below 0.5 are flipped).
* **Categorical features:** mutual information with the target.

In [ ]:
auc_numeric = pd.Series({c: roc_auc_score(df[TARGET], df[c]) for c in common.NUMERIC_RAW})
auc_numeric = ((auc_numeric - 0.5).abs() + 0.5).sort_values()

mi_categorical = pd.Series(
    {c: mutual_info_score(df[c].astype(str), df[TARGET]) for c in common.CATEGORICAL}
).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
auc_numeric.plot.barh(ax=axes[0], color="#8da0cb")
axes[0].set_title("Numeric features: univariate ROC-AUC")
axes[0].axvline(0.5, color="grey", linestyle="--")
mi_categorical.plot.barh(ax=axes[1], color="#fc8d62")
axes[1].set_title("Categorical features: mutual information with target")
fig.tight_layout()
plt.show()

## 3. Feature engineering
The EDA suggests a few cheap, interpretable features (computed inside the pipeline by `common.engineer`, so they are also applied at prediction time):

| feature | idea |
|---|---|
| `total_pages` | overall session length in pages |
| `total_duration` | overall time spent in the session |
| `avg_seconds_per_page` | engaged browsing vs. quick skimming |
| `product_page_share` | how much of the session was spent on product pages |
| `has_page_value` | indicator for `PageValues > 0`, the zero-inflated key signal |

In [ ]:
engineered = common.engineer(df[common.RAW_FEATURES])
display(engineered[common.ENGINEERED].describe().T.round(3))

pd.Series({c: roc_auc_score(df[TARGET], engineered[c]) for c in common.ENGINEERED}, name="univariate ROC-AUC").round(3).to_frame()

## 4. Train/test split and baseline models
I hold out a stratified **20 % test set** that is only used once, at the very end. All model selection and tuning uses **5-fold stratified cross-validation on the remaining 80 %**.
Four model families are compared: logistic regression (linear baseline), a decision tree, a random forest and gradient boosting (XGBoost). The baselines use the libraries' default hyper-parameters.

In [ ]:
X = df[common.RAW_FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
print(f"train: {len(X_train):,} sessions ({y_train.mean():.1%} purchases)")
print(f"test : {len(X_test):,} sessions ({y_test.mean():.1%} purchases)")

In [ ]:
baseline = {}
for name in common.MODEL_NAMES:
    baseline[name] = common.cv_scores(common.build_pipeline(name), X_train, y_train)
baseline_df = pd.DataFrame(baseline).T.sort_values("roc_auc_mean", ascending=False)
baseline_df.round(4)

## 5. Hyper-parameter tuning
Randomised search (5-fold CV, ROC-AUC) per model family. Lower the iteration counts if you want a faster run.

In [ ]:
N_ITER = {"logistic_regression": 15, "decision_tree": 40, "random_forest": 20, "xgboost": 40}

search_spaces = {
    "logistic_regression": {
        "model__C": loguniform(1e-3, 1e2),
        "model__class_weight": [None, "balanced"],
    },
    "decision_tree": {
        "model__max_depth": randint(3, 15),
        "model__min_samples_leaf": randint(5, 100),
        "model__class_weight": [None, "balanced"],
    },
    "random_forest": {
        "model__n_estimators": randint(150, 501),
        "model__max_depth": randint(4, 20),
        "model__min_samples_leaf": randint(1, 20),
        "model__max_features": ["sqrt", 0.3, 0.5],
        "model__class_weight": [None, "balanced_subsample"],
    },
    "xgboost": {
        "model__n_estimators": randint(100, 601),
        "model__learning_rate": loguniform(0.01, 0.3),
        "model__max_depth": randint(2, 8),
        "model__min_child_weight": randint(1, 20),
        "model__subsample": uniform(0.6, 0.4),
        "model__colsample_bytree": uniform(0.5, 0.5),
        "model__reg_lambda": loguniform(0.1, 20),
    },
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
searches = {}
for name in common.MODEL_NAMES:
    search = RandomizedSearchCV(
        common.build_pipeline(name),
        search_spaces[name],
        n_iter=N_ITER[name],
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        random_state=SEED,
    )
    search.fit(X_train, y_train)
    searches[name] = search
    print(f"{name:<20} best CV ROC-AUC = {search.best_score_:.4f}")

In [ ]:
best_params = {
    name: {k.removeprefix("model__"): v for k, v in s.best_params_.items()}
    for name, s in searches.items()
}
pd.Series({name: str(params) for name, params in common.to_builtin(best_params).items()}, name="best parameters").to_frame()

In [ ]:
tuned = {
    name: common.cv_scores(common.build_pipeline(name, best_params[name]), X_train, y_train)
    for name in common.MODEL_NAMES
}
comparison = pd.DataFrame(
    {
        "baseline_roc_auc": {n: baseline[n]["roc_auc_mean"] for n in common.MODEL_NAMES},
        "tuned_roc_auc": {n: tuned[n]["roc_auc_mean"] for n in common.MODEL_NAMES},
        "tuned_roc_auc_std": {n: tuned[n]["roc_auc_std"] for n in common.MODEL_NAMES},
        "tuned_pr_auc": {n: tuned[n]["pr_auc_mean"] for n in common.MODEL_NAMES},
    }
).sort_values("tuned_roc_auc", ascending=False)
display(comparison.round(4))

comparison[["baseline_roc_auc", "tuned_roc_auc"]].plot.bar(figsize=(8, 4), rot=0)
plt.ylabel("CV ROC-AUC")
plt.title("Baseline vs. tuned models (5-fold CV on the training set)")
plt.ylim(comparison["baseline_roc_auc"].min() - 0.03, comparison["tuned_roc_auc"].max() + 0.02)
plt.show()

The tuned CV scores are slightly optimistic, because the same folds were used to pick the hyper-parameters. The untouched test set below gives the unbiased estimate.

## 6. Final model, decision threshold and test-set evaluation
The model with the best tuned CV ROC-AUC is selected. A probability of 0.5 is a poor cut-off for an imbalanced problem, so the **threshold is chosen on out-of-fold predictions of the training set** (maximising F1) rather than on the test set.

In [ ]:
best_name = comparison.index[0]
print("Selected model:", best_name)
print("Parameters    :", common.to_builtin(best_params[best_name]))

final_pipe = common.build_pipeline(best_name, best_params[best_name])
oof = cross_val_predict(final_pipe, X_train, y_train, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]
threshold = common.best_f1_threshold(y_train, oof)
print(f"F1-optimal threshold on out-of-fold predictions: {threshold:.3f}")

In [ ]:
precision, recall, _ = precision_recall_curve(y_train, oof)
at_threshold = common.evaluate(y_train, oof, threshold)

plt.figure(figsize=(5.5, 4.5))
plt.plot(recall, precision)
plt.scatter([at_threshold["recall"]], [at_threshold["precision"]], color="red", zorder=3,
            label=f"chosen threshold = {threshold:.2f}")
plt.xlabel("recall")
plt.ylabel("precision")
plt.title("Out-of-fold precision-recall curve (train)")
plt.legend()
plt.show()

In [ ]:
final_pipe.fit(X_train, y_train)
proba_test = final_pipe.predict_proba(X_test)[:, 1]

results = pd.DataFrame(
    {
        "threshold 0.5": common.evaluate(y_test, proba_test, 0.5),
        f"tuned threshold ({threshold:.2f})": common.evaluate(y_test, proba_test, threshold),
    }
)
results.round(4)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
RocCurveDisplay.from_predictions(y_test, proba_test, ax=axes[0])
axes[0].set_title("ROC curve (test)")
PrecisionRecallDisplay.from_predictions(y_test, proba_test, ax=axes[1])
axes[1].set_title("Precision-recall curve (test)")
ConfusionMatrixDisplay.from_predictions(
    y_test, (proba_test >= threshold).astype(int), ax=axes[2],
    display_labels=["no purchase", "purchase"], colorbar=False,
)
axes[2].set_title(f"Confusion matrix (test, threshold {threshold:.2f})")
fig.tight_layout()
plt.show()

## 7. What drives the predictions?
**Permutation importance** on the test set: how much does ROC-AUC drop when one raw feature is shuffled? Because the pipeline starts from raw features, the engineered features are recomputed on every shuffle and their effect is attributed to the raw columns they come from.

In [ ]:
perm = permutation_importance(final_pipe, X_test, y_test, scoring="roc_auc", n_repeats=10,
                              random_state=SEED, n_jobs=-1)
perm_importance = pd.Series(perm.importances_mean, index=X_test.columns).sort_values()

plt.figure(figsize=(7, 6))
perm_importance.plot.barh(xerr=pd.Series(perm.importances_std, index=X_test.columns)[perm_importance.index], color="#8da0cb")
plt.xlabel("drop in ROC-AUC when shuffled")
plt.title(f"Permutation importance ({best_name})")
plt.show()

In [ ]:
model = final_pipe.named_steps["model"]
feature_names = final_pipe.named_steps["prep"].get_feature_names_out()
if hasattr(model, "feature_importances_"):
    native = pd.Series(model.feature_importances_, index=feature_names)
elif hasattr(model, "coef_"):
    native = pd.Series(model.coef_[0], index=feature_names)
else:
    native = None

if native is not None:
    top = native.reindex(native.abs().sort_values(ascending=False).index).head(15).iloc[::-1]
    plt.figure(figsize=(7, 6))
    top.plot.barh(color="#fc8d62")
    plt.title(f"Top 15 model-native feature scores ({best_name})")
    plt.show()

## 8. Export the tuned parameters
`train.py` reads its hyper-parameters from `params.json`. Saving the tuned values here closes the loop: **notebook -> `params.json` -> `train.py` -> `model.bin` -> web service**.

In [ ]:
with open(common.PARAMS_PATH, "w", encoding="utf-8") as f:
    json.dump(common.to_builtin(best_params), f, indent=2)
    f.write("\n")
print("Saved tuned parameters to", common.PARAMS_PATH.name)

## Conclusions and next steps
* The cleaned dataset has strongly imbalanced classes, so models were compared with ROC-AUC / PR-AUC and the decision threshold was tuned on out-of-fold predictions.
* Sections 2.6 and 7 show which features carry the signal. Check whether the behavioural features (page values, exit/bounce rates, time on product pages) dominate the technical and geographic codes, and whether the engineered features add anything.
* Re-run `python train.py` to rebuild `model.bin` with the tuned parameters, then rebuild the Docker image.

**Possible improvements:** probability calibration, cost-sensitive threshold (value of a recovered sale vs. cost of a discount), time-based validation (the data is only one year of traffic), and more features such as the page sequence of a session.